# Mouse kidney local inference from fixed-seed manual-alignment coordinates

This real-data notebook continues from the reported fixed-seed NL3/IL3 kidney alignment and runs post-alignment local inference with the current public `spalignde` API. The upstream Kidney workflow uses Leiden resolution `0.2`, seed `1000`, and a selected manual similarity pre-alignment (`scale=1`, `theta=0`, `translation_x=-36.20040965`, `translation_y=-153.38356513`) before 5,000 S-LDDMM iterations with `restore_best_checkpoint=False`. IL3 is the 2,965-spot query and NL3 is the unchanged 3,215-spot reference.

The raw count matrices and tissue-position files come from Zenodo record `17676992`. The checked-out repository supplies compact coordinate tables generated directly from `kidney_IL3_to_NL3_aligned.h5ad`, the output of the public manual-alignment notebook. The source H5AD and packaged coordinate SHA-256 values are recorded in the metadata.

The analysis proceeds through one continuous handoff:

1. load public expression data and the fixed-seed manual-alignment coordinates;
2. match spots by terminal Visium barcode;
3. construct the shared grid and local neighborhoods;
4. estimate stable-gene and density-based mismatch risk;
5. fit gene-specific mismatch-aware local tests with the public `fit_local_de` function; and
6. report grid-level statistics, FDR-adjusted regions, and representative gene maps.

NL3 defines the reference coordinate system and IL3 is the query. Users may replace the checked-in coordinates with their own spAlignDE output, but a custom alignment is not expected to reproduce the recorded numerical results unless its evaluated spot set and coordinates are identical.

**Fixed-seed reproducibility.** Upstream clustering and alignment use seed `1000`; this inference workflow uses seed `1` and `n_jobs=1`. For the closest numerical reproduction, set `PYTHONHASHSEED=1` before kernel startup and keep the documented input order, package version, and configuration fixed. Small floating-point differences in the last displayed digits can remain across numerical-library builds.

## Alignment-to-inference handoff

The recorded website example starts from the output of the public fixed-seed Kidney alignment notebook rather than rerunning alignment inside this inference notebook:

- upstream notebook: `cross_sample_alignment_mouse_kidney_alignment_nb.ipynb`;
- handoff artifact: `tutorials/cross_sample/kidney/output/kidney_IL3_to_NL3_aligned.h5ad`;
- evaluated coordinate columns: `x_aligned` and `y_aligned` for IL3 and the unchanged NL3 reference;
- coordinate scale: the saved compact array coordinates are multiplied by the recorded factor of 50 for inference.

The repository packages a compact, hash-tracked copy of this exact manual-alignment handoff as `src/spalignde/datasets/kidney/aligned_coords_IL3.csv.gz` and `aligned_coords_NL3.csv.gz`. The alignment input and region annotations come from STcompare record `20647680`; this inference stage reloads the NL3/IL3 10x matrices and tissue-position tables from source record `17676992` and joins them to the saved coordinates by terminal barcode. The two records therefore serve different handoff roles.

For a custom alignment, either point `SPALIGNDE_KIDNEY_ALIGNED_H5AD` to an H5AD containing `sample_id`, `x_aligned`, and `y_aligned`, or point `SPALIGNDE_ALIGNMENT_DIR` to a directory containing `aligned_coords_NL3.csv` and `aligned_coords_IL3.csv`. Set only one input variable. `SPALIGNDE_KIDNEY_DATA_DIR` and `SPALIGNDE_TUTORIAL_WORK_DIR` can redirect the raw-data and working directories.

```python
import os
os.environ["SPALIGNDE_KIDNEY_ALIGNED_H5AD"] = "/path/to/custom_alignment.h5ad"
```

Raw 10x matrices are always reloaded because an alignment object may contain only the gene-filtered matrix used for clustering and registration.

## 1. Install the package and tutorial dependencies

Install the editable package with its optional tutorial dependencies in the active environment:

```bash
python -m pip install -e ".[tutorial]"
```

The notebook imports the packaged fixed-seed alignment coordinates automatically. Only the public raw Visium files need to be downloaded separately.


In [1]:
from pathlib import Path
import json
import os
import random
import warnings


def find_repository_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src/spalignde/datasets/kidney").is_dir():
            return candidate
    raise FileNotFoundError(
        "Run this notebook from a spAlignDE checkout containing "
        "src/spalignde/datasets/kidney."
    )


REPOSITORY_ROOT = find_repository_root()
FIXED_ALIGNMENT_DIR = REPOSITORY_ROOT / "src/spalignde/datasets/kidney"
WORK_DIR = Path(
    os.environ.get(
        "SPALIGNDE_TUTORIAL_WORK_DIR",
        Path.cwd() / "spalignde_kidney_tutorial",
    )
).expanduser().resolve()
DATA_DIR = Path(
    os.environ.get("SPALIGNDE_KIDNEY_DATA_DIR", WORK_DIR / "raw")
).expanduser().resolve()
USER_ALIGNMENT_H5AD_VALUE = os.environ.get("SPALIGNDE_KIDNEY_ALIGNED_H5AD")
USER_ALIGNMENT_H5AD = (
    Path(USER_ALIGNMENT_H5AD_VALUE).expanduser().resolve()
    if USER_ALIGNMENT_H5AD_VALUE
    else None
)
USER_ALIGNMENT_DIR_VALUE = os.environ.get("SPALIGNDE_ALIGNMENT_DIR")
USER_ALIGNMENT_DIR = (
    Path(USER_ALIGNMENT_DIR_VALUE).expanduser().resolve()
    if USER_ALIGNMENT_DIR_VALUE
    else None
)
if USER_ALIGNMENT_H5AD is not None and USER_ALIGNMENT_DIR is not None:
    raise ValueError(
        "Set only one of SPALIGNDE_KIDNEY_ALIGNED_H5AD or "
        "SPALIGNDE_ALIGNMENT_DIR."
    )
os.environ.setdefault("MPLCONFIGDIR", str(WORK_DIR / ".matplotlib"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

import urllib.request

warnings.filterwarnings(
    "ignore",
    message="pkg_resources is deprecated as an API.*",
    category=UserWarning,
)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
from IPython.display import display

import spalignde
from spalignde import (
    cluster_trajectories,
    fit_local_de,
    gene_level_acat_pvalue,
    gene_level_age_trend_acat,
    plot_local_result,
    prepare_inference,
)
from spalignde.datasets import (
    build_visium_coordinate_table,
    canonical_visium_barcodes,
)

WORKFLOW_SEED = 1
random.seed(WORKFLOW_SEED)
np.random.seed(WORKFLOW_SEED)

alignment_metadata = json.loads(
    (FIXED_ALIGNMENT_DIR / "metadata.json").read_text()
)
alignment_source = (
    "upstream kidney alignment H5AD"
    if USER_ALIGNMENT_H5AD is not None
    else (
        "user-provided coordinate CSV files"
        if USER_ALIGNMENT_DIR is not None
        else "checked-in fixed-seed manual alignment"
    )
)
print("spalignde version:", spalignde.__version__)
print("Raw-data source: Zenodo record 17676992 (local cache configured)")
print("Aligned-coordinate source:", alignment_source)
print("Risk-map radius rule: 1.5 x shared-grid spacing")

spalignde version: 0.1.0
Raw-data source: Zenodo record 17676992 (local cache configured)
Aligned-coordinate source: checked-in fixed-seed manual alignment
Risk-map radius rule: 1.5 x shared-grid spacing


## 2. Obtain the raw Visium files

The source expression data are available from Zenodo record `17676992` under a CC BY 4.0 license. Set `DOWNLOAD_ZENODO=True` to download the four required files into `DATA_DIR`, or place them there manually. The much smaller precomputed alignment-coordinate files are installed with `spalignde` and require no separate download.


In [2]:
DOWNLOAD_ZENODO = False
ZENODO_BASE = "https://zenodo.org/records/17676992/files"
RAW_FILES = [
    "NL3_filtered_feature_bc_matrix.h5",
    "IL3_filtered_feature_bc_matrix.h5",
    "NL3_tissue_positions.csv",
    "IL3_tissue_positions.csv",
]

if DOWNLOAD_ZENODO:
    for filename in RAW_FILES:
        destination = DATA_DIR / filename
        if not destination.exists():
            url = f"{ZENODO_BASE}/{filename}?download=1"
            print("Downloading", url)
            urllib.request.urlretrieve(url, destination)

missing_raw = [name for name in RAW_FILES if not (DATA_DIR / name).exists()]
if missing_raw:
    raise FileNotFoundError(
        "Missing raw Visium files in DATA_DIR: "
        + ", ".join(missing_raw)
        + ". Download them from Zenodo record 17676992 or set "
        + "DOWNLOAD_ZENODO=True."
    )

if USER_ALIGNMENT_H5AD is not None and not USER_ALIGNMENT_H5AD.is_file():
    raise FileNotFoundError(
        "SPALIGNDE_KIDNEY_ALIGNED_H5AD does not exist: "
        + str(USER_ALIGNMENT_H5AD)
    )

if USER_ALIGNMENT_DIR is not None:
    missing_aligned = [
        name
        for name in ("aligned_coords_NL3.csv", "aligned_coords_IL3.csv")
        if not (USER_ALIGNMENT_DIR / name).exists()
    ]
    if missing_aligned:
        raise FileNotFoundError(
            "SPALIGNDE_ALIGNMENT_DIR is set but is missing: "
            + ", ".join(missing_aligned)
        )

print("Raw Visium inputs are present.")
print("Coordinate version:", alignment_metadata["coordinate_version"])

Raw Visium inputs are present.
Coordinate version: fixed-seed manual-prealignment cross-sample tutorial


## 3. Match raw spots to precomputed aligned coordinates

`build_visium_coordinate_table` is the package-level one-to-one handoff from
tissue positions and aligned coordinates to standardized `barcode`,
`cell_id`, `sample_id`, `x`, `y`, `x_aligned`, and `y_aligned` columns.
Matching uses terminal 10x barcodes and never row order.

In [3]:
upstream_h5ad_coordinates = None
if USER_ALIGNMENT_H5AD is not None:
    import anndata as ad

    aligned_h5ad = ad.read_h5ad(USER_ALIGNMENT_H5AD, backed="r")
    required_obs = {"sample_id", "x_aligned", "y_aligned"}
    missing_obs = required_obs.difference(aligned_h5ad.obs.columns)
    if missing_obs:
        aligned_h5ad.file.close()
        raise ValueError(
            "Upstream alignment H5AD is missing obs columns: "
            + ", ".join(sorted(missing_obs))
        )
    upstream_h5ad_coordinates = aligned_h5ad.obs[
        ["sample_id", "x_aligned", "y_aligned"]
    ].copy()
    upstream_h5ad_coordinates.insert(
        0, "cell_id", aligned_h5ad.obs_names.astype(str)
    )
    aligned_h5ad.file.close()
    coordinate_scale = float(alignment_metadata["coordinate_scale_factor"])
    upstream_h5ad_coordinates["x"] = (
        pd.to_numeric(upstream_h5ad_coordinates.pop("x_aligned"))
        * coordinate_scale
    )
    upstream_h5ad_coordinates["y"] = (
        pd.to_numeric(upstream_h5ad_coordinates.pop("y_aligned"))
        * coordinate_scale
    )

coordinate_tables = []
for sample_id in ("NL3", "IL3"):
    if upstream_h5ad_coordinates is not None:
        aligned = upstream_h5ad_coordinates.loc[
            upstream_h5ad_coordinates["sample_id"].astype(str) == sample_id
        ].copy()
    elif USER_ALIGNMENT_DIR is not None:
        aligned = pd.read_csv(
            USER_ALIGNMENT_DIR / f"aligned_coords_{sample_id}.csv"
        )
    else:
        aligned = pd.read_csv(
            FIXED_ALIGNMENT_DIR / f"aligned_coords_{sample_id}.csv.gz"
        )
    if "cell_id" not in aligned.columns and "barcode" in aligned.columns:
        aligned = aligned.rename(columns={"barcode": "cell_id"})
    positions = pd.read_csv(DATA_DIR / f"{sample_id}_tissue_positions.csv")
    coordinates = build_visium_coordinate_table(
        positions,
        aligned,
        sample_id=sample_id,
    )
    coordinate_tables.append(coordinates)
    coordinate_range = coordinates[["x_aligned", "y_aligned"]].agg(["min", "max"])
    print(sample_id, "matched spots:", len(coordinates))
    display(coordinate_range)

coordinate_data = pd.concat(coordinate_tables, ignore_index=True)

NL3 matched spots: 3215


,x_aligned,y_aligned
min,350,0
max,3250,6350


IL3 matched spots: 2965


,x_aligned,y_aligned
min,361.325836,-67.005516
max,3325.536133,6344.030762


## 4. Build the inference table

The current inference package validates the standardized coordinates with `build_visium_coordinate_table`. The code below then reloads the raw 10x matrices, computes per-gene support across the pair, and joins expression to coordinates by terminal barcode. These are data-handoff steps only; calibration, age-trend testing, and trajectory selection are never reimplemented in the notebook.

The kidney analysis retains genes detected in at least 10 spots with at least 10 total counts across the pair. Spot-wise library-size normalization to 10,000 is applied later inside `prepare_inference`.

In [4]:
TARGET_LIBRARY_SIZE = 10_000
MIN_DETECTED_SPOTS = 10
MIN_TOTAL_COUNTS = 10
REQUESTED_GENES = ["Cbr1", "Cd44", "Myo5a"]


def read_count_matrix(sample_id):
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="Variable names are not unique.*",
            category=UserWarning,
        )
        adata = sc.read_10x_h5(
            DATA_DIR / f"{sample_id}_filtered_feature_bc_matrix.h5",
            gex_only=True,
        )
    adata.var_names_make_unique()
    adata.obs_names = canonical_visium_barcodes(
        adata.obs_names,
        source_name=f"{sample_id} count-matrix barcodes",
    ).to_numpy()
    if adata.obs_names.duplicated().any():
        raise ValueError(f"{sample_id} count matrix contains duplicate barcodes.")
    return adata


counts = {sample_id: read_count_matrix(sample_id) for sample_id in ("NL3", "IL3")}
common_genes = pd.Index(counts["NL3"].var_names).intersection(
    pd.Index(counts["IL3"].var_names)
)


def gene_support(adata, genes):
    matrix = adata[:, genes].X
    detected = np.asarray((matrix > 0).sum(axis=0)).reshape(-1)
    total = np.asarray(matrix.sum(axis=0)).reshape(-1)
    return detected, total


nl3_detected, nl3_total = gene_support(counts["NL3"], common_genes)
il3_detected, il3_total = gene_support(counts["IL3"], common_genes)
keep = (
    (nl3_detected + il3_detected >= MIN_DETECTED_SPOTS)
    & (nl3_total + il3_total >= MIN_TOTAL_COUNTS)
)
risk_genes = common_genes[keep].tolist()
genes_to_test = [gene for gene in REQUESTED_GENES if gene in common_genes]
missing_requested = sorted(set(REQUESTED_GENES) - set(genes_to_test))
if missing_requested:
    raise ValueError(f"Requested genes were not found: {missing_requested}")
risk_genes = list(dict.fromkeys([*risk_genes, *genes_to_test]))


def make_sample_table(sample_id):
    adata = counts[sample_id]
    coordinates = (
        coordinate_data.loc[coordinate_data["sample_id"].eq(sample_id)]
        .set_index("barcode")
    )
    missing_expression = coordinates.index.difference(adata.obs_names)
    if len(missing_expression):
        raise ValueError(
            f"{sample_id} has {len(missing_expression)} aligned spots without expression."
        )
    ordered_barcodes = coordinates.index.tolist()
    matrix = adata[ordered_barcodes, risk_genes].X
    if sparse.issparse(matrix):
        matrix = matrix.toarray()
    expression = pd.DataFrame(
        np.asarray(matrix, dtype=np.float32),
        index=ordered_barcodes,
        columns=risk_genes,
    )
    table = coordinates.join(expression, how="inner")
    table.index.name = "barcode"
    table["batch"] = "kidney_pair"
    return table.reset_index()


inference_data = pd.concat(
    [make_sample_table("NL3"), make_sample_table("IL3")],
    ignore_index=True,
)

print("Inference table shape:", inference_data.shape)
print("Genes to test:", genes_to_test)
print("Risk-gene candidates:", len(risk_genes), "of", len(common_genes), "shared genes")
display(inference_data.head())

Inference table shape: (6180, 16454)
Genes to test: ['Cbr1', 'Cd44', 'Myo5a']
Risk-gene candidates: 16446 of 32285 shared genes


,barcode,cell_id,sample_id,x,y,x_aligned,y_aligned,Sox17,Mrpl15,Lypla1,...,mt-Cytb,CAAA01118383.1,Csprs,Vamp7,Spry3,Tmlhe,CR974586.4,CAAA01147332.1,AC149090.1,batch
0,AAACAAGTATCTCCCA-1,NL3__AAACAAGTATCTCCCA-1,NL3,50,102,2500.0,5100.0,1.0,1.0,8.0,...,351.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,kidney_pair
1,AAACAGAGCGACTCCT-1,NL3__AAACAGAGCGACTCCT-1,NL3,14,94,700.0,4700.0,0.0,0.0,5.0,...,111.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,kidney_pair
2,AAACAGCTTTCAGAAG-1,NL3__AAACAGCTTTCAGAAG-1,NL3,43,9,2150.0,450.0,0.0,1.0,0.0,...,45.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,kidney_pair
3,AAACAGGGTCTATATT-1,NL3__AAACAGGGTCTATATT-1,NL3,47,13,2350.0,650.0,0.0,0.0,1.0,...,230.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,kidney_pair
4,AAACATTTCCCGGATT-1,NL3__AAACATTTCCCGGATT-1,NL3,61,97,3050.0,4850.0,0.0,0.0,2.0,...,433.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,kidney_pair


## 5. Prepare and fit local inference

NL3 defines the reference coordinate system and IL3 is the query. `prepare_inference` constructs the shared grid, neighborhoods, stable-gene distributional risk, and density channel. Its R-driven candidate resolution is retained when the estimated number of tissue-valid grid locations lies between the median per-sample spot count, $N_{typ}$, and $2N_{typ}$; otherwise the tissue-occupancy-aware resolution is moved toward the corresponding bound. Passing `grid_n=` explicitly overrides this automatic resolution rule. Here `density_energy_share=0.75` sets the density channel's target share of standardized mismatch-feature energy; it is not a direct weight on the final risk map or variance.

For each gene and contrast, `fit_local_de` first obtains local statistics without mismatch inflation. It bins them by normalized local risk, median-centers each bin, divides its MAD by the Student-t null MAD, constrains nonnegative excess dispersion to be nondecreasing with risk, and fits a quadratic through the origin with bounded rescaling at the retained bin nearest the 80th risk percentile. A provisional coefficient is valid only when the within-contrast calibration succeeds, uses at least `max(500, 4 * min_bin_n)` locations, retains at least four distinct risk bins with positive-risk support, has finite fit and rescaling quantities, and returns a finite nonnegative capped local coefficient with zero global coefficient. Failed contrasts are excluded; a successful zero remains valid.

Across multiple contrasts, valid provisional coefficients are combined by an equal-weight Huber center to obtain one gene-specific coefficient shared by all contrasts. With this single kidney contrast, the Huber center equals its valid provisional coefficient. The final variance factor is `1 + lambda_g * risk**2`, so zero-risk locations retain their base variance. The comparison-level global risk score remains diagnostic only. Cell-type adjustment is disabled here because the public inputs lack validated cell-type labels.

In [5]:
prepared = prepare_inference(
    inference_data,
    reference="NL3",
    genes=genes_to_test,
    risk_genes=risk_genes,
    aligned_coordinate_key=("x_aligned", "y_aligned"),
    cell_type_key=None,
    density_energy_share=0.75,
    library_size=TARGET_LIBRARY_SIZE,
    n_jobs=1,
    random_state=WORKFLOW_SEED,
)

result = fit_local_de(
    prepared,
    genes=genes_to_test,
    contrast="vs_reference",
    mismatch_aware=True,
    technical_adjustment=True,
    cell_type_adjustment=False,
    global_offset=False,
    region_cleanup=False,
    n_jobs=1,
    random_state=WORKFLOW_SEED,
    verbose=True,
)

calibration_rows = []
for gene in genes_to_test:
    terrain = result.fits[gene]["terrain_data"]
    assert "Wv_by_time" in terrain and "use_by_time" in terrain
    calibration = terrain["risk_calibration"]
    assert (
        calibration["method"]
        == "per_contrast_local_calibration_then_equal_weight_huber"
    )
    calibration_rows.append(
        {
            "gene": gene,
            "method": calibration["method"],
            "total contrasts": calibration["n_contrasts_total"],
            "valid contrasts": calibration["n_contrasts_valid"],
            "valid time IDs": calibration["valid_time_ids"],
            "invalid time IDs": calibration["invalid_time_ids"],
            "lambda_g": calibration["lambda_local_hat"],
        }
    )
    provisional_summary = {
        str(time_id): {
            "valid": detail["valid"],
            "validity_reasons": detail["validity_reasons"],
            "lambda_local_hat": detail["lambda_local_hat"],
        }
        for time_id, detail in calibration["provisional_by_time"].items()
    }
    print(gene, "provisional coefficients:", provisional_summary)
    print(
        gene,
        "aggregation:",
        {
            key: calibration["aggregation"][key]
            for key in (
                "method",
                "n_valid",
                "valid_time_ids",
                "invalid_time_ids",
                "lambda_local_hat",
            )
        },
    )

print("Reference:", prepared.reference)
print("Query samples:", prepared.shared["time_ids"])
print("Shared grid locations:", len(prepared.shared["grid_eval"]))
display(pd.DataFrame(calibration_rows))

[INFO] detecting tissue geometry on 6180 spots ...


[INFO] geometry ready: 0 holes, 6187 grid points, R=191.9, h_grid=75.49, R_map=113.2


[INFO] building KNN trees for 2 samples ...


[SWND] N_A=3215, N_B=2965, G=16446


[marker-screen] var_thr=0.00675775 | delta_thr=0.05 | cor_min=0.2 | n_var=8223 | n_delta=11046 | n_cor=2664 | n_all=632


[risk-calib-aggregate] valid=1/1 method=Huber(kappa=1.345) lambda_local=15.0518
[risk-calib-aggregate] valid=1/1 method=Huber(kappa=1.345) lambda_local=0.557915
[risk-calib-aggregate] valid=1/1 method=Huber(kappa=1.345) lambda_local=5.41691
[batch_run] completed 3 / 3 genes
Cbr1 provisional coefficients: {'IL3': {'valid': True, 'validity_reasons': (), 'lambda_local_hat': 15.051838385690003}}
Cbr1 aggregation: {'method': 'equal_weight_huber_location', 'n_valid': 1, 'valid_time_ids': ('IL3',), 'invalid_time_ids': (), 'lambda_local_hat': 15.051838385690003}
Cd44 provisional coefficients: {'IL3': {'valid': True, 'validity_reasons': (), 'lambda_local_hat': 0.5579152230168773}}
Cd44 aggregation: {'method': 'equal_weight_huber_location', 'n_valid': 1, 'valid_time_ids': ('IL3',), 'invalid_time_ids': (), 'lambda_local_hat': 0.5579152230168773}
Myo5a provisional coefficients: {'IL3': {'valid': True, 'validity_reasons': (), 'lambda_local_hat': 5.416914897600465}}
Myo5a aggregation: {'method': 'eq

,gene,method,total contrasts,valid contrasts,valid time IDs,invalid time IDs,lambda_g
0,Cbr1,per_contrast_local_calibration_then_equal_weig...,1,1,"(IL3,)",(),15.051838
1,Cd44,per_contrast_local_calibration_then_equal_weig...,1,1,"(IL3,)",(),0.557915
2,Myo5a,per_contrast_local_calibration_then_equal_weig...,1,1,"(IL3,)",(),5.416915


## 6. Inspect local results

BH adjustment is performed separately for each tested gene and the IL3-versus-NL3 contrast across valid shared-grid locations. The table reports grid-level calls and statistic summaries together with the gene-level ACAT P value obtained by combining retained raw local P values across valid grid locations; adjusted q-values are not substituted into ACAT. This is an omnibus gene-level P value, not a local P value or a genome-wide FDR-adjusted gene discovery value. Each figure shows observed NL3 and IL3 expression on a shared expression scale together with the zero-centered local statistic. Because `region_cleanup=False`, red contours directly trace connected components of the `q < 0.05` grid mask.


In [6]:
summary_rows = []
for gene, fit in result.fits.items():
    gene_acat_pvalue = gene_level_acat_pvalue(result, gene)
    terrain = fit["terrain_data"]
    for query_id in terrain["time_ids"]:
        statistic = np.asarray(terrain["stat_by_time"][query_id], dtype=float)
        q_value = np.asarray(terrain["q_by_time"][query_id], dtype=float)
        significant = np.asarray(
            terrain["sig_mask_by_time"][query_id],
            dtype=bool,
        )
        summary_rows.append(
            {
                "gene": gene,
                "contrast": f"{query_id} - NL3",
                "gene-level ACAT P value": gene_acat_pvalue,
                "significant grid locations": int(significant.sum()),
                "minimum q-value": float(np.nanmin(q_value)),
                "median |t|": float(np.nanmedian(np.abs(statistic))),
            }
        )

display(pd.DataFrame(summary_rows))

for gene in genes_to_test:
    figure = plot_local_result(
        result,
        gene,
        show_expression=True,
        invert_y=False,
    )
    plt.show()


,gene,contrast,gene-level ACAT P value,significant grid locations,minimum q-value,median |t|
0,Cbr1,IL3 - NL3,1.676437e-14,2954,7.780664e-32,2.435333
1,Cd44,IL3 - NL3,4.099926e-06,1452,1.018210e-05,1.747929
2,Myo5a,IL3 - NL3,7.355228e-14,2268,6.605387e-21,1.946836


## 7. Interpretation and extensions

This example demonstrates the complete alignment-output $\rightarrow$ shared-grid inference $\rightarrow$ local DE maps $\rightarrow$ gene-level ACAT P value handoff. It is a single matched-section NL3-versus-IL3 comparison and should not be interpreted as replicate-level population inference. `SPALIGNDE_KIDNEY_ALIGNED_H5AD` continues directly from the public fixed-seed kidney alignment notebook; the packaged coordinates reproduce that same validated handoff by default, and `SPALIGNDE_ALIGNMENT_DIR` remains available for standardized coordinate CSV files. Users with reliable cell-type annotations can add a `celltype` column and enable `cell_type_adjustment=True`. Multiple query sections can be analyzed against one reference and summarized with gene-level ACAT or trajectory clustering.
